# Experiment 3.0.1 — Single-$\tau_{syn}$ × objective comparison

## Formal protocol: `paired_split_v2`

At a fixed 3-layer feature SNN, which homogeneous synaptic time scale is best, and does the answer depend on the training objective?

Backbone: `30 -> 128 -> 128 -> 64`, no recurrence, no bias in the three SNN Linear layers. L1/L2/L3 use the same `shift_syn` within one run. Shifts: `[2, 3, 4, 5, 6, 7]`.

Objectives:
- `timestep_ce`: shared `64 -> 12` classifier at every valid timestep; CE averaged over valid timesteps.
- `relative10_sequence_ce`: valid gesture divided into 10 relative-progress bins; L3 spike counts are flattened and classified once per segment.
- `fixed250_sequence_ce`: valid-masked fixed 250 ms bins (16 samples at 64 Hz); L3 spike counts are flattened and classified once per segment. SNN state is continuous across bin boundaries.

Grid: `6 shifts × 3 objectives × 3 seeds = 54 runs`, seeds `(11, 23, 101)`, fixed split seed `12345`.

### `paired_split_v2` controls
- user split exactly matches the existing Phase-B / Experiment-3.0 rule: direct `default_rng(12345)`, sorted users, floor train/validation fractions;
- for a given master seed, every shift and every objective receives identical initial SNN backbone weights;
- objective-specific heads are initialized deterministically after the paired backbone;
- checkpoints carry and validate protocol, split, architecture, and training provenance;
- firing rates use total valid spikes divided by total valid neuron-timesteps;
- zero-input firing is measured without invoking classification metrics.

Formal training is performed by `scripts/bash_script/SNN_Bash/run_exp_3_0_1_cpu_array.bash`. This notebook is analysis-only. Earlier root-level Experiment 3.0.1 artifacts are pilot results and are intentionally not loaded here.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_repo_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'snn').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('writingRing repository root not found')

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.experiment_3_0_1_single_tau_objectives import (
    EXPERIMENT_ID, PROTOCOL_VERSION, EXPECTED_RUNS,
    SHIFTS, OBJECTIVES, SEEDS, WIDTHS, protocol_results_dir,
)

RESULTS_DIR = protocol_results_dir(REPO_ROOT)
RESULTS_PATH = RESULTS_DIR / 'experiment_3_0_1_results.csv'
HISTORY_PATH = RESULTS_DIR / 'experiment_3_0_1_history.csv'
SUMMARY_PATH = RESULTS_DIR / 'experiment_3_0_1_summary.csv'

for path in (RESULTS_PATH, HISTORY_PATH, SUMMARY_PATH):
    if not path.exists():
        raise FileNotFoundError(
            f'Formal {PROTOCOL_VERSION} artifact missing: {path}. '
            'Run the Slurm array and finalizer first.'
        )

RESULTS = pd.read_csv(RESULTS_PATH)
HISTORY = pd.read_csv(HISTORY_PATH)
SUMMARY = pd.read_csv(SUMMARY_PATH)

if len(RESULTS) != EXPECTED_RUNS:
    raise RuntimeError(f'Expected {EXPECTED_RUNS} runs, found {len(RESULTS)}')
if set(RESULTS.protocol_version.unique()) != {PROTOCOL_VERSION}:
    raise RuntimeError('Results contain the wrong protocol version')

print('Repository root:', REPO_ROOT)
print('Experiment:', EXPERIMENT_ID)
print('Protocol:', PROTOCOL_VERSION)
print('Results dir:', RESULTS_DIR.relative_to(REPO_ROOT))
print('Widths:', WIDTHS)
print('Shifts:', SHIFTS)
print('Objectives:', OBJECTIVES)
print('Seeds:', SEEDS)
print('Completed runs:', len(RESULTS), '/', EXPECTED_RUNS)
display(SUMMARY)

## 1. Per-run results

In [ ]:
display(RESULTS.sort_values(['objective', 'shift', 'seed']))

## 2. Native test balanced accuracy vs. shift

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for objective in OBJECTIVES:
    part = SUMMARY[SUMMARY.objective == objective].sort_values('shift')
    ax.errorbar(
        part['shift'], part['mean_test_balanced_accuracy'],
        yerr=part['sd_test_balanced_accuracy'],
        marker='o', capsize=3, label=objective,
    )
ax.set(
    xlabel='homogeneous shift_syn in L1/L2/L3',
    ylabel='test balanced accuracy',
    xticks=SHIFTS,
)
ax.grid(alpha=0.25)
ax.legend()
plt.show()

## 3. Common fixed-250-ms frozen linear-probe BA

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for objective in OBJECTIVES:
    part = SUMMARY[SUMMARY.objective == objective].sort_values('shift')
    ax.errorbar(
        part['shift'], part['mean_probe_test_balanced_accuracy'],
        yerr=part['sd_probe_test_balanced_accuracy'],
        marker='o', capsize=3, label=objective,
    )
ax.set(
    xlabel='homogeneous shift_syn in L1/L2/L3',
    ylabel='common fixed250 probe test BA',
    xticks=SHIFTS,
)
ax.grid(alpha=0.25)
ax.legend()
plt.show()

## 4. Validation BA vs. epoch

In [ ]:
mean_history = (
    HISTORY.groupby(['objective', 'shift', 'epoch'], as_index=False)
    .val_balanced_accuracy.mean()
)
for objective in OBJECTIVES:
    fig, ax = plt.subplots(figsize=(8, 5))
    part = mean_history[mean_history.objective == objective]
    for shift in SHIFTS:
        curve = part[part['shift'] == shift]
        ax.plot(curve.epoch, curve.val_balanced_accuracy, label=f'shift {shift}')
    ax.set(title=objective, xlabel='epoch', ylabel='mean validation BA')
    ax.grid(alpha=0.25)
    ax.legend(ncol=2)
    plt.show()

## 5. Firing-rate diagnostics

In [ ]:
fr_cols = [
    'objective', 'shift', 'seed',
    'test_l1_firing_rate', 'test_l2_firing_rate', 'test_l3_firing_rate',
    'zero_l1_firing_rate', 'zero_l2_firing_rate', 'zero_l3_firing_rate',
]
display(RESULTS[fr_cols])

for objective in OBJECTIVES:
    fig, ax = plt.subplots(figsize=(8, 5))
    part = (
        RESULTS[RESULTS.objective == objective]
        .groupby('shift', as_index=False)[
            ['test_l1_firing_rate','test_l2_firing_rate','test_l3_firing_rate']
        ].mean()
    )
    for column, label in [
        ('test_l1_firing_rate', 'L1'),
        ('test_l2_firing_rate', 'L2'),
        ('test_l3_firing_rate', 'L3'),
    ]:
        ax.plot(part['shift'], part[column], marker='o', label=label)
    ax.set(
        title=objective, xlabel='shift_syn',
        ylabel='valid spikes / neuron / timestep', xticks=SHIFTS,
    )
    ax.grid(alpha=0.25)
    ax.legend()
    plt.show()

## 6. Generalization gap and checkpoint epoch

In [ ]:
display(
    SUMMARY[[
        'objective', 'shift', 'tau_syn_ms',
        'mean_train_test_ba_gap', 'sd_train_test_ba_gap',
        'mean_best_epoch', 'sd_best_epoch',
    ]].sort_values(['objective', 'shift'])
)

## 7. Rankings

In [ ]:
native = (
    RESULTS.groupby(['objective','shift']).test_balanced_accuracy
    .agg(['mean','std']).sort_values('mean', ascending=False)
)
probe = (
    RESULTS.groupby(['objective','shift']).probe_test_balanced_accuracy
    .agg(['mean','std']).sort_values('mean', ascending=False)
)
print('Native head ranking')
display(native)
print('Common frozen-probe ranking')
display(probe)

## Interpretation
Use the 3-seed mean and SD, not one seed. First ask whether a consistent single-$\tau$ optimum exists. Then ask whether the optimum changes with supervision. Agreement between native-head BA and the common fixed250 probe supports a representation-level effect; disagreement suggests the objective-specific head/readout contributes materially. Inspect firing rates and the train-test gap before carrying a single-$\tau$ baseline into the multi-$\tau$ architecture study.